# Addresses26 — Free Automated Agent Discovery v2

**Purpose:** build a free, conservative, auditable estate-agent attribution pilot for BN postcode Land Registry transactions.

This notebook improves the earlier pilot by:

- removing `<NA>` / `nan` artefacts from address strings
- avoiding over-strict price-based queries by default
- using short, realistic search queries
- using free search via `duckduckgo_search` / `ddgs` when available
- adding known-agent matching for Sussex / BN area agents
- saving raw search evidence, extracted candidates, agent tables, and summaries

**Important:** HM Land Registry Price Paid Data does **not** contain estate-agent data. Agent attribution here is probabilistic and must remain separate from verified Land Registry transaction data.

In [8]:
# ============================================================
# 0. Install optional free-search package
# ============================================================
# Colab may already have this unavailable; install quietly if needed.
# If installation fails, the notebook will still create the query plan.

import sys, subprocess, importlib.util

packages_to_try = ["duckduckgo_search", "ddgs"]
for pkg in packages_to_try:
    if importlib.util.find_spec(pkg) is None:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        except Exception as e:
            print(f"Could not install {pkg}: {e}")

In [9]:
# ============================================================
# 1. Mount Drive and configure paths
# ============================================================
from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np
import re, json, time, hashlib, textwrap
from datetime import datetime, timezone

try:
    drive.mount('/content/drive')
except Exception as e:
    print('Drive mount warning:', e)

BASE = Path('/content/drive/MyDrive/Addresses26_Data')
DERIVED = BASE / 'derived' / 'land_registry_price_paid'
OUT = BASE / 'reports' / 'agent_attribution_free_v2'
OUT.mkdir(parents=True, exist_ok=True)

# Configurable pilot controls
POSTCODE_PREFIX = 'BN'
YEARS = [2021, 2022, 2023, 2024, 2025]
MAX_TRANSACTIONS_THIS_RUN = 25      # increase after first successful pilot
MAX_RESULTS_PER_QUERY = 5
SLEEP_SECONDS_BETWEEN_QUERIES = 8   # keep slow and polite
USE_PRICE_QUERY = False             # keep False unless results are too broad
RANDOM_SEED = 26

print('BASE:', BASE)
print('DERIVED:', DERIVED)
print('OUT:', OUT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE: /content/drive/MyDrive/Addresses26_Data
DERIVED: /content/drive/MyDrive/Addresses26_Data/derived/land_registry_price_paid
OUT: /content/drive/MyDrive/Addresses26_Data/reports/agent_attribution_free_v2


In [10]:
# ============================================================
# 2. Load verified yearly Land Registry parquet files
# ============================================================
frames = []
missing_files = []
for y in YEARS:
    p = DERIVED / f'pp_bn_standard_{y}.parquet'
    if p.exists():
        df = pd.read_parquet(p)
        df['sale_year'] = y
        frames.append(df)
        print(f'Loaded {y}: {len(df):,} rows')
    else:
        missing_files.append(str(p))

if not frames:
    raise FileNotFoundError('No yearly parquet files found. Check DERIVED path and filenames.')

sales = pd.concat(frames, ignore_index=True)
print('Combined rows:', len(sales))
if missing_files:
    print('Missing files:', missing_files)

# Standardise column names just in case
sales.columns = [str(c).strip().lower() for c in sales.columns]
sales.head()

Loaded 2021: 18,279 rows
Loaded 2022: 14,934 rows
Loaded 2023: 11,515 rows
Loaded 2024: 12,532 rows
Loaded 2025: 11,425 rows
Combined rows: 68685


,transaction_id,price,date_of_transfer,postcode,property_type,old_new,duration,paon,saon,street,...,ppd_category_type,record_status,postcode_area,postcode_district,postcode_sector,property_type_label,old_new_label,duration_label,address_compact,sale_year
0,{F87E72F8-D9D5-176C-E053-6B04A8C0D2BE},650000,2021-01-03,BN3 5DL,S,N,F,58,<NA>,PORTLAND ROAD,...,A,A,BN,<NA>,None,Semi-detached,Established,Freehold,"58, PORTLAND ROAD, HOVE",2021
1,{BC8936BB-74DB-0E2C-E053-6C04A8C0DBF4},265000,2021-01-04,BN1 3RT,F,N,L,1,SECOND FLOOR FLAT,WEST HILL ROAD,...,A,A,BN,<NA>,None,Flat/Maisonette,Established,Leasehold,"SECOND FLOOR FLAT, 1, WEST HILL ROAD, BRIGHTON",2021
2,{BC8936BB-F58B-0E2C-E053-6C04A8C0DBF4},505000,2021-01-04,BN16 2EA,D,N,F,45,<NA>,GLENVILLE ROAD,...,A,A,BN,<NA>,None,Detached,Established,Freehold,"45, GLENVILLE ROAD, RUSTINGTON, LITTLEHAMPTON",2021
3,{BC8936BB-743D-0E2C-E053-6C04A8C0DBF4},430000,2021-01-04,BN2 5AD,F,N,L,"THE LEAS, 34 - 35",FLAT 1,SUSSEX SQUARE,...,A,A,BN,<NA>,None,Flat/Maisonette,Established,Leasehold,"FLAT 1, THE LEAS, 34 - 35, SUSSEX SQUARE, BRIG...",2021
4,{BC8936BB-6FA6-0E2C-E053-6C04A8C0DBF4},800000,2021-01-04,BN20 8DL,D,N,F,25,<NA>,OLD CAMP ROAD,...,A,A,BN,<NA>,None,Detached,Established,Freehold,"25, OLD CAMP ROAD, EASTBOURNE",2021


In [11]:
# ============================================================
# 3. Required columns and robust address helpers
# ============================================================
required_any = ['postcode']
missing = [c for c in required_any if c not in sales.columns]
if missing:
    raise AssertionError(f'Missing required columns: {missing}')

# Land Registry common columns are usually:
# transaction_unique_identifier, price, date_of_transfer, postcode, property_type,
# old_new, duration, paon, saon, street, locality, town_city, district, county, ppd_category_type, record_status

for c in ['paon', 'saon', 'street', 'locality', 'town_city', 'postcode', 'price', 'date_of_transfer']:
    if c not in sales.columns:
        sales[c] = pd.NA

def clean_token(x):
    if pd.isna(x):
        return ''
    s = str(x).strip()
    bad = {'', '<NA>', '<na>', 'NA', 'N/A', 'nan', 'NaN', 'None', 'NONE', 'null', 'NULL'}
    if s in bad:
        return ''
    # Remove repeated whitespace and odd angle-bracket NA leftovers
    s = re.sub(r'<\s*NA\s*>', '', s, flags=re.I)
    s = re.sub(r'\s+', ' ', s).strip(' ,')
    return s

def clean_address_from_row(row):
    # SAON can be flat/unit; PAON is primary name/number. Put SAON first only if it looks meaningful.
    parts = [
        clean_token(row.get('saon')),
        clean_token(row.get('paon')),
        clean_token(row.get('street')),
        clean_token(row.get('locality')),
        clean_token(row.get('town_city')),
    ]
    parts = [p for p in parts if p]
    # Deduplicate adjacent repeated values
    out = []
    for p in parts:
        if not out or p.upper() != out[-1].upper():
            out.append(p)
    return ' '.join(out)

def make_transaction_id(row):
    for c in ['transaction_unique_identifier', 'transaction_id', 'uid']:
        v = clean_token(row.get(c))
        if v:
            return v
    base = '|'.join([
        clean_token(row.get('price')),
        clean_token(row.get('date_of_transfer')),
        clean_token(row.get('postcode')),
        clean_address_from_row(row)
    ])
    return hashlib.sha1(base.encode('utf-8')).hexdigest()[:16]

sales['address_clean'] = sales.apply(clean_address_from_row, axis=1)
sales['postcode_clean'] = sales['postcode'].astype(str).str.upper().str.strip()
sales['town_clean'] = sales['town_city'].apply(clean_token)
sales['sale_date'] = pd.to_datetime(sales['date_of_transfer'], errors='coerce')
sales['sale_year'] = sales['sale_date'].dt.year.fillna(sales['sale_year']).astype('Int64')
sales['transaction_id'] = sales.apply(make_transaction_id, axis=1)
sales['postcode_district'] = sales['postcode_clean'].str.extract(r'^([A-Z]{1,2}\d{1,2}[A-Z]?)', expand=False)

# Drop rows with unusable address/postcode
sales = sales[(sales['postcode_clean'].str.startswith(POSTCODE_PREFIX, na=False)) & (sales['address_clean'].str.len() > 3)].copy()

print('Usable sales rows:', len(sales))
sales[['transaction_id','sale_year','address_clean','postcode_clean','price','sale_date']].head(10)

Usable sales rows: 68685


,transaction_id,sale_year,address_clean,postcode_clean,price,sale_date
0,{F87E72F8-D9D5-176C-E053-6B04A8C0D2BE},2021,58 PORTLAND ROAD HOVE,BN3 5DL,650000,2021-01-03
1,{BC8936BB-74DB-0E2C-E053-6C04A8C0DBF4},2021,SECOND FLOOR FLAT 1 WEST HILL ROAD BRIGHTON,BN1 3RT,265000,2021-01-04
2,{BC8936BB-F58B-0E2C-E053-6C04A8C0DBF4},2021,45 GLENVILLE ROAD RUSTINGTON LITTLEHAMPTON,BN16 2EA,505000,2021-01-04
3,{BC8936BB-743D-0E2C-E053-6C04A8C0DBF4},2021,"FLAT 1 THE LEAS, 34 - 35 SUSSEX SQUARE BRIGHTON",BN2 5AD,430000,2021-01-04
4,{BC8936BB-6FA6-0E2C-E053-6C04A8C0DBF4},2021,25 OLD CAMP ROAD EASTBOURNE,BN20 8DL,800000,2021-01-04
5,{BC8936BB-F642-0E2C-E053-6C04A8C0DBF4},2021,4 HAWTH GROVE SEAFORD,BN25 2RP,370000,2021-01-04
6,{BEF7EBBF-7F92-7A76-E053-6B04A8C092F7},2021,43A SOUTHDOWN ROAD SHOREHAM-BY-SEA,BN43 5AL,242500,2021-01-04
7,{BEF7EBBF-807E-7A76-E053-6B04A8C092F7},2021,40 PENLANDS VALE STEYNING,BN44 3PL,441200,2021-01-04
8,{BC8936BB-72FA-0E2C-E053-6C04A8C0DBF4},2021,FLAT 12 48 - 50 DYKE ROAD BRIGHTON,BN1 3JB,230000,2021-01-05
9,{BEF7EBBF-4E0E-7A76-E053-6B04A8C092F7},2021,102 ELDRED AVENUE BRIGHTON,BN1 5EH,497500,2021-01-05


In [12]:
# ============================================================
# 4. Create a balanced sample for this run
# ============================================================
# We sample across years and postcode districts to avoid only central Brighton/Hove dominating.

rng = np.random.default_rng(RANDOM_SEED)

# Start with a district/year shuffled frame
sample_source = sales.copy()
sample_source['_rand'] = rng.random(len(sample_source))
sample_source = sample_source.sort_values(['sale_year', 'postcode_district', '_rand'])

# Take a roughly balanced year sample
per_year = max(1, MAX_TRANSACTIONS_THIS_RUN // len(YEARS))
samples = []
for y, g in sample_source.groupby('sale_year'):
    if int(y) in YEARS:
        samples.append(g.head(per_year))

pilot = pd.concat(samples, ignore_index=True) if samples else sample_source.head(MAX_TRANSACTIONS_THIS_RUN)

# If under target, top up from remaining rows
if len(pilot) < MAX_TRANSACTIONS_THIS_RUN:
    used = set(pilot['transaction_id'])
    topup = sample_source[~sample_source['transaction_id'].isin(used)].head(MAX_TRANSACTIONS_THIS_RUN - len(pilot))
    pilot = pd.concat([pilot, topup], ignore_index=True)

pilot = pilot.head(MAX_TRANSACTIONS_THIS_RUN).copy()
print('Transactions to search this run:', len(pilot))
display(pilot.groupby('sale_year').size().reset_index(name='rows'))
display(pilot[['transaction_id','sale_year','postcode_district','address_clean','postcode_clean','price']].head(20))

pilot.to_csv(OUT / 'search_transaction_sample.csv', index=False)

Transactions to search this run: 25


,sale_year,rows
0,2021,5
1,2022,5
2,2023,5
3,2024,5
4,2025,5


,transaction_id,sale_year,postcode_district,address_clean,postcode_clean,price
0,{C6209F5F-26CB-295E-E053-6C04A8C0DDCC},2021,BN1,14 TONGDEAN RISE BRIGHTON,BN1 5JG,700000
1,{FFA361DA-BDBE-8A03-E053-4804A8C01F61},2021,BN1,FLAT 13 AVALON WEST STREET BRIGHTON,BN1 2RP,361000
2,{C6209F5F-290C-295E-E053-6C04A8C0DDCC},2021,BN1,50B INWOOD CRESCENT BRIGHTON,BN1 5AQ,337500
3,{C18F412B-4FCB-81A6-E053-6B04A8C0AD18},2021,BN1,71 REDHILL DRIVE BRIGHTON,BN1 5FL,580000
4,{C3C3F9B5-A223-362B-E053-6B04A8C03ACC},2021,BN1,50 DOVER ROAD BRIGHTON,BN1 6LN,640000
5,{EA3278AA-7D26-2676-E053-6B04A8C015F8},2022,BN1,27 LARKFIELD WAY BRIGHTON,BN1 8EG,700000
6,{E53EDD2E-3F3E-83EC-E053-6B04A8C03A59},2022,BN1,27 CLEVELAND ROAD BRIGHTON,BN1 6FF,875000
7,{F3B6C198-0CDB-6E40-E053-6C04A8C0B3B4},2022,BN1,GROUND FLOOR FLAT 15 STAFFORD ROAD BRIGHTON,BN1 5PE,425000
8,{E53EDD2D-9C96-83EC-E053-6B04A8C03A59},2022,BN1,FLAT 1 35 TEMPLE STREET BRIGHTON,BN1 3BH,388500
9,{E2D14905-5A41-4C2D-E053-6B04A8C0422B},2022,BN1,19A UPPER HOLLINGDEAN ROAD BRIGHTON,BN1 7GA,330000


In [13]:
# ============================================================
# 5. Known-agent dictionary for candidate extraction
# ============================================================
# This list is intentionally editable. Add/remove local agents as the project learns.

KNOWN_AGENTS = sorted(set([
    # National / regional chains
    'Fox & Sons', 'Connells', 'Cubitt & West', 'King & Chasemore', 'Hamptons', 'Savills',
    'Mishon Mackay', 'Mishons', 'Oakley Property', 'Brand Vaughan', 'Leaders', 'Jacobs Steel',
    'Robert Luff', 'Michael Jones', 'Sawyers', 'Austin Gray', 'Phillips and Still',
    'Kendrick Property Services', 'Coapt', 'John Hilton', 'Goldin Lemcke', 'Move Revolution',
    'Purplebricks', 'Yopa', 'Strike', 'Fine & Country', 'Jackson-Stops', 'Your Move',
    'Mansell McTaggart', 'Freeman Forman', 'Taylor Robinson', 'Choices', 'Martin & Co',
    'Belvoir', 'Hunters', 'Open House', 'Priors', 'Pearson Keehan', 'Lextons',
    # Brighton/Hove/Lewes/Worthing/Eastbourne/Bognor-ish local names
    'David Maslen', 'Maslen Estate Agents', 'Wheeler’s Estate Agents', 'Wheelers Estate Agents',
    'Harringtons Lettings', 'Spencer & Leigh', 'Clifford Sales & Lettings', 'Paul Bott & Company',
    'Nicholas James', 'Elliotts', 'Foster & Co', 'Goldin Lemcke', 'Healy and Newsom',
    'Pearson Keehan', 'Lextons', 'Foster & Co', 'Hyman Hill', 'Warwick Baker', 'Bacon and Company',
    'Matthew Anthony', 'Aspire Residential', 'James & James', 'Stafford Johnson',
    'Coast & Country', 'Rowland Gorringe', 'Lampons', 'Phillip Mann', 'Charles Cox',
    'Home Sweet Home', 'Town Property', 'Taylor Engley', 'Rager & Roberts', 'Reid & Dean',
    'Ancells Estates', 'Open House Estate Agents', 'Phillip Mann Estate Agents',
]))

import re

AGENT_PATTERNS = [
    r"(?i)marketed by[:\s]+([A-Za-z0-9&.,'’\- ]{2,80})",
    r"(?i)estate agent[s]?[:\s]+([A-Za-z0-9&.,'’\- ]{2,80})",
    r"(?i)sold by[:\s]+([A-Za-z0-9&.,'’\- ]{2,80})",
    r"(?i)listed by[:\s]+([A-Za-z0-9&.,'’\- ]{2,80})",
]

def normalise_agent_name(name):
    s = clean_token(name)
    s = re.sub(r'\s+', ' ', s).strip(' -–—|,.;:')
    # Stop at common trailing junk
    s = re.split(r'(?:for sale|sold price|property|rightmove|zoopla|onthemarket|details|branch)', s, flags=re.I)[0].strip(' -–—|,.;:')
    return s[:120]

def extract_agent_candidates(text):
    text = clean_token(text)
    if not text:
        return []
    found = []
    low = text.lower()
    for agent in KNOWN_AGENTS:
        if agent.lower() in low:
            found.append({'agent_name': agent, 'method': 'known_agent_dictionary'})
    for pat in AGENT_PATTERNS:
        for m in re.finditer(pat, text):
            candidate = normalise_agent_name(m.group(1))
            if candidate and len(candidate) >= 3:
                found.append({'agent_name': candidate, 'method': 'phrase_regex'})
    # Deduplicate by lowercase name
    dedup = {}
    for x in found:
        k = x['agent_name'].lower()
        if k not in dedup:
            dedup[k] = x
    return list(dedup.values())

print('Known agents:', len(KNOWN_AGENTS))

Known agents: 71


In [14]:
# ============================================================
# 6. Query builder v2 — no <NA>, no default price over-constraint
# ============================================================

def format_price_for_query(price):
    try:
        return f"£{int(float(price)):,}"
    except Exception:
        return ''

def build_queries(row):
    address = clean_token(row.get('address_clean'))
    postcode = clean_token(row.get('postcode_clean'))
    town = clean_token(row.get('town_clean'))
    year = clean_token(row.get('sale_year'))
    price = format_price_for_query(row.get('price'))

    # Prefer shorter, realistic queries.
    queries = []
    if address and postcode:
        queries.append(f'"{address}" "{postcode}" estate agent')
        queries.append(f'"{address}" "{postcode}" "marketed by"')
        queries.append(f'"{address}" "{postcode}" property for sale')
    if postcode and town:
        queries.append(f'"{postcode}" "{town}" "sold" "estate agent"')
    if USE_PRICE_QUERY and postcode and price:
        queries.append(f'"{postcode}" "{price}" "estate agent" "{year}"')

    # Remove duplicates and any accidentally contaminated NA strings
    clean_queries = []
    seen = set()
    for q in queries:
        q = re.sub(r'\s+', ' ', q).replace('<NA>', '').strip()
        if q and q.lower() not in seen:
            seen.add(q.lower())
            clean_queries.append(q)
    return clean_queries

pilot['queries'] = pilot.apply(build_queries, axis=1)
for i, row in pilot.head(5).iterrows():
    print(row['transaction_id'], row['address_clean'], row['postcode_clean'])
    for q in row['queries']:
        print('  query:', q)

{C6209F5F-26CB-295E-E053-6C04A8C0DDCC} 14 TONGDEAN RISE BRIGHTON BN1 5JG
  query: "14 TONGDEAN RISE BRIGHTON" "BN1 5JG" estate agent
  query: "14 TONGDEAN RISE BRIGHTON" "BN1 5JG" "marketed by"
  query: "14 TONGDEAN RISE BRIGHTON" "BN1 5JG" property for sale
  query: "BN1 5JG" "BRIGHTON" "sold" "estate agent"
{FFA361DA-BDBE-8A03-E053-4804A8C01F61} FLAT 13 AVALON WEST STREET BRIGHTON BN1 2RP
  query: "FLAT 13 AVALON WEST STREET BRIGHTON" "BN1 2RP" estate agent
  query: "FLAT 13 AVALON WEST STREET BRIGHTON" "BN1 2RP" "marketed by"
  query: "FLAT 13 AVALON WEST STREET BRIGHTON" "BN1 2RP" property for sale
  query: "BN1 2RP" "BRIGHTON" "sold" "estate agent"
{C6209F5F-290C-295E-E053-6C04A8C0DDCC} 50B INWOOD CRESCENT BRIGHTON BN1 5AQ
  query: "50B INWOOD CRESCENT BRIGHTON" "BN1 5AQ" estate agent
  query: "50B INWOOD CRESCENT BRIGHTON" "BN1 5AQ" "marketed by"
  query: "50B INWOOD CRESCENT BRIGHTON" "BN1 5AQ" property for sale
  query: "BN1 5AQ" "BRIGHTON" "sold" "estate agent"
{C18F412B-4FCB-

In [15]:
# ============================================================
# 7. Free search adapter
# ============================================================
# Uses ddgs/duckduckgo_search if available. This may be rate-limited or unavailable.
# The notebook remains useful because it always saves the query plan.

SEARCH_BACKEND = None

try:
    from ddgs import DDGS
    SEARCH_BACKEND = 'ddgs'
except Exception:
    try:
        from duckduckgo_search import DDGS
        SEARCH_BACKEND = 'duckduckgo_search'
    except Exception as e:
        SEARCH_BACKEND = None
        print('No free search backend available:', e)

print('SEARCH_BACKEND:', SEARCH_BACKEND)

query_plan_rows = []
for _, row in pilot.iterrows():
    for q in row['queries']:
        query_plan_rows.append({
            'transaction_id': row['transaction_id'],
            'sale_year': int(row['sale_year']) if not pd.isna(row['sale_year']) else None,
            'postcode_district': row['postcode_district'],
            'address_clean': row['address_clean'],
            'postcode': row['postcode_clean'],
            'price': row.get('price'),
            'query': q,
        })
query_plan = pd.DataFrame(query_plan_rows)
query_plan.to_csv(OUT / 'agent_discovery_query_plan.csv', index=False)
print('Query plan rows:', len(query_plan))
display(query_plan.head(10))

SEARCH_BACKEND: ddgs
Query plan rows: 100


,transaction_id,sale_year,postcode_district,address_clean,postcode,price,query
0,{C6209F5F-26CB-295E-E053-6C04A8C0DDCC},2021,BN1,14 TONGDEAN RISE BRIGHTON,BN1 5JG,700000,"""14 TONGDEAN RISE BRIGHTON"" ""BN1 5JG"" estate a..."
1,{C6209F5F-26CB-295E-E053-6C04A8C0DDCC},2021,BN1,14 TONGDEAN RISE BRIGHTON,BN1 5JG,700000,"""14 TONGDEAN RISE BRIGHTON"" ""BN1 5JG"" ""markete..."
2,{C6209F5F-26CB-295E-E053-6C04A8C0DDCC},2021,BN1,14 TONGDEAN RISE BRIGHTON,BN1 5JG,700000,"""14 TONGDEAN RISE BRIGHTON"" ""BN1 5JG"" property..."
3,{C6209F5F-26CB-295E-E053-6C04A8C0DDCC},2021,BN1,14 TONGDEAN RISE BRIGHTON,BN1 5JG,700000,"""BN1 5JG"" ""BRIGHTON"" ""sold"" ""estate agent"""
4,{FFA361DA-BDBE-8A03-E053-4804A8C01F61},2021,BN1,FLAT 13 AVALON WEST STREET BRIGHTON,BN1 2RP,361000,"""FLAT 13 AVALON WEST STREET BRIGHTON"" ""BN1 2RP..."
5,{FFA361DA-BDBE-8A03-E053-4804A8C01F61},2021,BN1,FLAT 13 AVALON WEST STREET BRIGHTON,BN1 2RP,361000,"""FLAT 13 AVALON WEST STREET BRIGHTON"" ""BN1 2RP..."
6,{FFA361DA-BDBE-8A03-E053-4804A8C01F61},2021,BN1,FLAT 13 AVALON WEST STREET BRIGHTON,BN1 2RP,361000,"""FLAT 13 AVALON WEST STREET BRIGHTON"" ""BN1 2RP..."
7,{FFA361DA-BDBE-8A03-E053-4804A8C01F61},2021,BN1,FLAT 13 AVALON WEST STREET BRIGHTON,BN1 2RP,361000,"""BN1 2RP"" ""BRIGHTON"" ""sold"" ""estate agent"""
8,{C6209F5F-290C-295E-E053-6C04A8C0DDCC},2021,BN1,50B INWOOD CRESCENT BRIGHTON,BN1 5AQ,337500,"""50B INWOOD CRESCENT BRIGHTON"" ""BN1 5AQ"" estat..."
9,{C6209F5F-290C-295E-E053-6C04A8C0DDCC},2021,BN1,50B INWOOD CRESCENT BRIGHTON,BN1 5AQ,337500,"""50B INWOOD CRESCENT BRIGHTON"" ""BN1 5AQ"" ""mark..."


In [16]:
# ============================================================
# 8. Run slow free web discovery
# ============================================================
# Safe defaults: 25 transactions, max 5 results per query, slow pause.

raw_results = []

if SEARCH_BACKEND is None:
    print('No search backend available. Query plan saved only.')
else:
    with DDGS() as ddgs:
        total = len(query_plan)
        for idx, r in query_plan.iterrows():
            print(f"[{idx+1}/{total}] {r['transaction_id']} :: {r['query']}")
            try:
                # region='uk-en' works for some versions; fallback if unsupported
                try:
                    results = list(ddgs.text(r['query'], region='uk-en', max_results=MAX_RESULTS_PER_QUERY))
                except TypeError:
                    results = list(ddgs.text(r['query'], max_results=MAX_RESULTS_PER_QUERY))
                for rank, item in enumerate(results, start=1):
                    raw_results.append({
                        'transaction_id': r['transaction_id'],
                        'query': r['query'],
                        'rank': rank,
                        'title': item.get('title') or item.get('heading') or '',
                        'href': item.get('href') or item.get('url') or '',
                        'body': item.get('body') or item.get('snippet') or '',
                        'searched_utc': datetime.now(timezone.utc).isoformat(),
                    })
            except Exception as e:
                raw_results.append({
                    'transaction_id': r['transaction_id'],
                    'query': r['query'],
                    'rank': None,
                    'title': '',
                    'href': '',
                    'body': '',
                    'error': str(e),
                    'searched_utc': datetime.now(timezone.utc).isoformat(),
                })
            time.sleep(SLEEP_SECONDS_BETWEEN_QUERIES)

raw = pd.DataFrame(raw_results)
raw_path = OUT / 'agent_discovery_raw_search_results.csv'
raw.to_csv(raw_path, index=False)
print('Saved raw results:', raw_path, 'rows:', len(raw))
display(raw.head(10))

[1/100] {C6209F5F-26CB-295E-E053-6C04A8C0DDCC} :: "14 TONGDEAN RISE BRIGHTON" "BN1 5JG" estate agent
[2/100] {C6209F5F-26CB-295E-E053-6C04A8C0DDCC} :: "14 TONGDEAN RISE BRIGHTON" "BN1 5JG" "marketed by"
[3/100] {C6209F5F-26CB-295E-E053-6C04A8C0DDCC} :: "14 TONGDEAN RISE BRIGHTON" "BN1 5JG" property for sale
[4/100] {C6209F5F-26CB-295E-E053-6C04A8C0DDCC} :: "BN1 5JG" "BRIGHTON" "sold" "estate agent"
[5/100] {FFA361DA-BDBE-8A03-E053-4804A8C01F61} :: "FLAT 13 AVALON WEST STREET BRIGHTON" "BN1 2RP" estate agent
[6/100] {FFA361DA-BDBE-8A03-E053-4804A8C01F61} :: "FLAT 13 AVALON WEST STREET BRIGHTON" "BN1 2RP" "marketed by"
[7/100] {FFA361DA-BDBE-8A03-E053-4804A8C01F61} :: "FLAT 13 AVALON WEST STREET BRIGHTON" "BN1 2RP" property for sale
[8/100] {FFA361DA-BDBE-8A03-E053-4804A8C01F61} :: "BN1 2RP" "BRIGHTON" "sold" "estate agent"
[9/100] {C6209F5F-290C-295E-E053-6C04A8C0DDCC} :: "50B INWOOD CRESCENT BRIGHTON" "BN1 5AQ" estate agent
[10/100] {C6209F5F-290C-295E-E053-6C04A8C0DDCC} :: "50B INWOOD

,transaction_id,query,rank,title,href,body,searched_utc,error
0,{C6209F5F-26CB-295E-E053-6C04A8C0DDCC},"""14 TONGDEAN RISE BRIGHTON"" ""BN1 5JG"" estate a...",1.0,"Houses for sale & to rent in BN1 5JG, Tongdean...",https://housesforsaletorent.co.uk/houses/engla...,Properties in BN1 5JG have an average house pr...,2026-04-27T17:58:04.262226+00:00,NaN
1,{C6209F5F-26CB-295E-E053-6C04A8C0DDCC},"""14 TONGDEAN RISE BRIGHTON"" ""BN1 5JG"" estate a...",2.0,"Interesting Information for Tongdean Rise, Bri...",https://www.streetcheck.co.uk/postcode/bn15jg,Tongdean Rise in Brighton is in the South East...,2026-04-27T17:58:04.262262+00:00,NaN
2,{C6209F5F-26CB-295E-E053-6C04A8C0DDCC},"""14 TONGDEAN RISE BRIGHTON"" ""BN1 5JG"" estate a...",3.0,Explore Tongdean Rise in Brighton in BN1 5JG,https://www.streetlist.co.uk/bn/bn1/bn1-5/tong...,"October 27, 2017 - Our data shows that Tongdea...",2026-04-27T17:58:04.262275+00:00,NaN
3,{C6209F5F-26CB-295E-E053-6C04A8C0DDCC},"""14 TONGDEAN RISE BRIGHTON"" ""BN1 5JG"" estate a...",4.0,"House Prices in Tongdean Rise, Brighton, East ...",https://www.rightmove.co.uk/house-prices/bn1/t...,"Sold House Prices in Tongdean Rise, Brighton, ...",2026-04-27T17:58:04.262286+00:00,NaN
4,{C6209F5F-26CB-295E-E053-6C04A8C0DDCC},"""14 TONGDEAN RISE BRIGHTON"" ""BN1 5JG"" estate a...",5.0,"House prices in Tongdean Rise, Brighton, East ...",https://www.zoopla.co.uk/house-prices/east-sus...,"Properties for sale in Tongdean Rise, Brighton...",2026-04-27T17:58:04.262299+00:00,NaN
5,{C6209F5F-26CB-295E-E053-6C04A8C0DDCC},"""14 TONGDEAN RISE BRIGHTON"" ""BN1 5JG"" ""markete...",1.0,"Tongdean Rise, Brighton, BN1 5JG - detailed in...",https://streetscan.co.uk/postcode/bn1-5jg,"View information about Tongdean Rise, Brighton...",2026-04-27T17:58:14.912619+00:00,NaN
6,{C6209F5F-26CB-295E-E053-6C04A8C0DDCC},"""14 TONGDEAN RISE BRIGHTON"" ""BN1 5JG"" ""markete...",2.0,Explore Tongdean Rise in Brighton in BN1 5JG,https://www.streetlist.co.uk/bn/bn1/bn1-5/tong...,"October 27, 2017 - Address: LIONSDENE, 11 THE ...",2026-04-27T17:58:14.912657+00:00,NaN
7,{C6209F5F-26CB-295E-E053-6C04A8C0DDCC},"""14 TONGDEAN RISE BRIGHTON"" ""BN1 5JG"" ""markete...",3.0,"Houses for sale & to rent in BN1 5JG, Tongdean...",https://housesforsaletorent.co.uk/houses/engla...,Properties in BN1 5JG have an average house pr...,2026-04-27T17:58:14.912669+00:00,NaN
8,{C6209F5F-26CB-295E-E053-6C04A8C0DDCC},"""14 TONGDEAN RISE BRIGHTON"" ""BN1 5JG"" ""markete...",4.0,Information about the BN1 5JG Postcode for Ton...,https://www.geopunk.co.uk/postcode/BN1-5JG,"Information about the BN1 5JG postcode, Tongde...",2026-04-27T17:58:14.912680+00:00,NaN
9,{C6209F5F-26CB-295E-E053-6C04A8C0DDCC},"""14 TONGDEAN RISE BRIGHTON"" ""BN1 5JG"" ""markete...",5.0,"House prices in Tongdean Rise, Brighton, East ...",https://www.zoopla.co.uk/house-prices/east-sus...,"Properties for sale in Tongdean Rise, Brighton...",2026-04-27T17:58:14.912692+00:00,NaN


In [19]:
# ============================================================
# 9. Extract candidate agents and score confidence
# ============================================================

def domain_from_url(url):
    s = clean_token(url).lower()
    m = re.search(r'https?://(?:www\.)?([^/]+)', s)
    return m.group(1) if m else ''

def score_candidate(row, agent_method):
    score = 0
    title = clean_token(row.get('title'))
    body = clean_token(row.get('body'))
    url = clean_token(row.get('href'))
    query = clean_token(row.get('query'))
    combined = f'{title} {body}'.lower()
    domain = domain_from_url(url)

    if agent_method == 'known_agent_dictionary': score += 35
    if agent_method == 'phrase_regex': score += 25
    if 'marketed by' in combined: score += 25
    if 'sold by' in combined: score += 20
    if 'estate agent' in combined: score += 10
    if any(d in domain for d in ['rightmove', 'zoopla', 'onthemarket', 'prime-location', 'primelocation']): score += 15
    if any(d in domain for d in ['property', 'estate', 'agent']): score += 5
    if 'postcode' in combined or re.search(r'BN\d', combined, flags=re.I): score += 5
    if row.get('rank') in [1, 2]: score += 5
    return min(score, 100)

def confidence_label(score):
    if score >= 75: return 'high_candidate'
    if score >= 50: return 'medium_candidate'
    if score >= 30: return 'low_candidate'
    return 'weak_signal'

candidate_rows = []

if "raw" in globals() and len(raw):
    for _, rr in raw.iterrows():
        title = str(rr.get("title", "") or "")
        snippet = str(rr.get("snippet", "") or "")
        body = str(rr.get("body", "") or "")

        evidence_text = f"{title} {snippet} {body}".strip()

        for cand in extract_agent_candidates(evidence_text):
            score = score_candidate(rr, cand["method"])
            candidate_rows.append({
                "transaction_id": rr.get("transaction_id"),
                "agent_name_raw": cand["agent_name"],
                "agent_name_norm": normalise_agent_name(cand["agent_name"]).lower(),
                "extraction_method": cand["method"],
                "confidence_score": score,
                "confidence": confidence_label(score),
                "source_url": rr.get("href"),
                "source_title": rr.get("title"),
                "source_snippet": rr.get("body"),
                "query": rr.get("query"),
                "rank": rr.get("rank"),
                "created_utc": datetime.now(timezone.utc).isoformat(),
            })

candidates = pd.DataFrame(candidate_rows)

if len(candidates):
    candidates = candidates.sort_values(
        ["transaction_id", "confidence_score"],
        ascending=[True, False]
    )

cand_path = OUT / "agent_candidates_extracted.csv"
candidates.to_csv(cand_path, index=False)

print("Candidate rows:", len(candidates))
display(candidates.head(20))
candidates = pd.DataFrame(candidate_rows)
if len(candidates):
    candidates = candidates.sort_values(['transaction_id','confidence_score'], ascending=[True, False])


Candidate rows: 2


,transaction_id,agent_name_raw,agent_name_norm,extraction_method,confidence_score,confidence,source_url,source_title,source_snippet,query,rank,created_utc
0,{01EB45EF-AE0F-40F3-E063-4704A8C05FDE},Spencer & Leigh,spencer & leigh,known_agent_dictionary,60,medium_candidate,https://media.onthemarket.com/properties/41601...,VebraAlto.com - Agency Cloud,"44 The Priory, London Road, Patcham, Brighton ...","""44 THE PRIORY LONDON ROAD PATCHAM BRIGHTON"" ""...",1.0,2026-04-27T18:28:10.396105+00:00
1,{01EB45EF-AE0F-40F3-E063-4704A8C05FDE},Spencer & Leigh,spencer & leigh,known_agent_dictionary,60,medium_candidate,https://media.onthemarket.com/properties/41601...,VebraAlto.com - Agency Cloud,"44 The Priory, London Road, Patcham, Brighton ...","""44 THE PRIORY LONDON ROAD PATCHAM BRIGHTON"" ""...",1.0,2026-04-27T18:28:10.396720+00:00


In [20]:
# ============================================================
# 10. Build canonical agent and transaction-agent tables
# ============================================================

def canonical_agent_id(name_norm):
    return hashlib.sha1(str(name_norm).encode('utf-8')).hexdigest()[:12]

if len(candidates):
    # One best candidate per transaction/agent/source may remain as evidence; choose top candidate per transaction for link table.
    candidates['agent_id'] = candidates['agent_name_norm'].apply(canonical_agent_id)

    agents = (candidates
              .groupby(['agent_id','agent_name_norm'], as_index=False)
              .agg(
                  agent_name_raw_example=('agent_name_raw','first'),
                  evidence_rows=('agent_id','size'),
                  max_confidence_score=('confidence_score','max'),
                  first_seen_utc=('created_utc','min'),
                  last_seen_utc=('created_utc','max')
              ))

    # Best candidate per transaction
    agent_transactions = (candidates
                          .sort_values(['transaction_id','confidence_score'], ascending=[True, False])
                          .drop_duplicates('transaction_id')
                          .copy())
    agent_transactions = agent_transactions[[
        'transaction_id','agent_id','agent_name_raw','agent_name_norm','confidence_score','confidence',
        'extraction_method','source_url','source_title','source_snippet','query','created_utc'
    ]]

else:
    agents = pd.DataFrame(columns=['agent_id','agent_name_norm','agent_name_raw_example','evidence_rows','max_confidence_score'])
    agent_transactions = pd.DataFrame(columns=['transaction_id','agent_id','agent_name_raw','confidence_score','confidence'])

agents_path_csv = OUT / 'agents_free_discovery.csv'
links_path_csv = OUT / 'agent_transactions_free_discovery.csv'
agents_path_parquet = OUT / 'agents_free_discovery.parquet'
links_path_parquet = OUT / 'agent_transactions_free_discovery.parquet'

agents.to_csv(agents_path_csv, index=False)
agent_transactions.to_csv(links_path_csv, index=False)
try:
    agents.to_parquet(agents_path_parquet, index=False)
    agent_transactions.to_parquet(links_path_parquet, index=False)
except Exception as e:
    print('Parquet save warning:', e)

print('Agents:', len(agents))
print('Agent transaction links:', len(agent_transactions))
display(agents.head(20))
display(agent_transactions.head(20))

Agents: 1
Agent transaction links: 1


,agent_id,agent_name_norm,agent_name_raw_example,evidence_rows,max_confidence_score,first_seen_utc,last_seen_utc
0,e83fffafa575,spencer & leigh,Spencer & Leigh,2,60,2026-04-27T18:28:10.396105+00:00,2026-04-27T18:28:10.396720+00:00


,transaction_id,agent_id,agent_name_raw,agent_name_norm,confidence_score,confidence,extraction_method,source_url,source_title,source_snippet,query,created_utc
0,{01EB45EF-AE0F-40F3-E063-4704A8C05FDE},e83fffafa575,Spencer & Leigh,spencer & leigh,60,medium_candidate,known_agent_dictionary,https://media.onthemarket.com/properties/41601...,VebraAlto.com - Agency Cloud,"44 The Priory, London Road, Patcham, Brighton ...","""44 THE PRIORY LONDON ROAD PATCHAM BRIGHTON"" ""...",2026-04-27T18:28:10.396105+00:00


In [21]:
# ============================================================
# 11. Summaries and hit-rate diagnostics
# ============================================================

summary = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'postcode_prefix': POSTCODE_PREFIX,
    'years': YEARS,
    'transactions_sampled': int(len(pilot)),
    'query_plan_rows': int(len(query_plan)),
    'raw_search_result_rows': int(len(raw)) if 'raw' in globals() else 0,
    'candidate_rows': int(len(candidates)),
    'unique_agents': int(len(agents)),
    'transactions_with_candidate_agent': int(agent_transactions['transaction_id'].nunique()) if len(agent_transactions) else 0,
    'hit_rate_transactions': float((agent_transactions['transaction_id'].nunique() / len(pilot)) if len(pilot) else 0),
    'search_backend': SEARCH_BACKEND,
    'max_transactions_this_run': MAX_TRANSACTIONS_THIS_RUN,
    'max_results_per_query': MAX_RESULTS_PER_QUERY,
    'sleep_seconds_between_queries': SLEEP_SECONDS_BETWEEN_QUERIES,
    'use_price_query': USE_PRICE_QUERY,
}

with open(OUT / 'agent_discovery_manifest.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

if len(agent_transactions):
    enriched = pilot.merge(agent_transactions[['transaction_id','agent_id','agent_name_raw','confidence_score','confidence']], on='transaction_id', how='left')
else:
    enriched = pilot.copy()
    for c in ['agent_id','agent_name_raw','confidence_score','confidence']:
        enriched[c] = pd.NA

enriched.to_csv(OUT / 'sample_with_agent_candidates.csv', index=False)

by_year = (enriched.assign(has_candidate=enriched['agent_name_raw'].notna())
           .groupby('sale_year', as_index=False)
           .agg(sample_rows=('transaction_id','count'), candidate_rows=('has_candidate','sum')))
by_year['hit_rate'] = by_year['candidate_rows'] / by_year['sample_rows']
by_year.to_csv(OUT / 'summary_by_year.csv', index=False)

by_district = (enriched.assign(has_candidate=enriched['agent_name_raw'].notna())
               .groupby('postcode_district', as_index=False)
               .agg(sample_rows=('transaction_id','count'), candidate_rows=('has_candidate','sum')))
by_district['hit_rate'] = by_district['candidate_rows'] / by_district['sample_rows']
by_district = by_district.sort_values(['candidate_rows','sample_rows'], ascending=[False, False])
by_district.to_csv(OUT / 'summary_by_postcode_district.csv', index=False)

print('Summary by year')
display(by_year)
print('Summary by postcode district')
display(by_district.head(30))

{
  "created_utc": "2026-04-27T18:28:22.527027+00:00",
  "postcode_prefix": "BN",
  "years": [
    2021,
    2022,
    2023,
    2024,
    2025
  ],
  "transactions_sampled": 25,
  "query_plan_rows": 100,
  "raw_search_result_rows": 151,
  "candidate_rows": 2,
  "unique_agents": 1,
  "transactions_with_candidate_agent": 1,
  "hit_rate_transactions": 0.04,
  "search_backend": "ddgs",
  "max_transactions_this_run": 25,
  "max_results_per_query": 5,
  "sleep_seconds_between_queries": 8,
  "use_price_query": false
}
Summary by year


,sale_year,sample_rows,candidate_rows,hit_rate
0,2021,5,0,0.0
1,2022,5,0,0.0
2,2023,5,1,0.2
3,2024,5,0,0.0
4,2025,5,0,0.0


Summary by postcode district


,postcode_district,sample_rows,candidate_rows,hit_rate
0,BN1,25,1,0.04


In [22]:
# ============================================================
# 12. Review best evidence rows
# ============================================================
# Inspect these rows to see whether the free-search evidence is useful enough to scale.

review_cols = [
    'transaction_id','agent_name_raw','confidence_score','confidence','source_title','source_url','source_snippet','query'
]
if len(candidates):
    display(candidates[review_cols].head(50))
else:
    print('No candidates extracted. Try one or more of:')
    print('- Increase MAX_TRANSACTIONS_THIS_RUN')
    print('- Increase MAX_RESULTS_PER_QUERY')
    print('- Add more local estate agents to KNOWN_AGENTS')
    print('- Set USE_PRICE_QUERY=True only if results are too broad')
    print('- Try a smaller district-specific sample, e.g. BN1/BN3/BN7')

,transaction_id,agent_name_raw,confidence_score,confidence,source_title,source_url,source_snippet,query
0,{01EB45EF-AE0F-40F3-E063-4704A8C05FDE},Spencer & Leigh,60,medium_candidate,VebraAlto.com - Agency Cloud,https://media.onthemarket.com/properties/41601...,"44 The Priory, London Road, Patcham, Brighton ...","""44 THE PRIORY LONDON ROAD PATCHAM BRIGHTON"" ""..."
1,{01EB45EF-AE0F-40F3-E063-4704A8C05FDE},Spencer & Leigh,60,medium_candidate,VebraAlto.com - Agency Cloud,https://media.onthemarket.com/properties/41601...,"44 The Priory, London Road, Patcham, Brighton ...","""44 THE PRIORY LONDON ROAD PATCHAM BRIGHTON"" ""..."


## Interpretation Guide

### If hit rate is 0–5%
The free search route is not finding enough indexed historical listing evidence. Improve the known-agent dictionary, simplify queries further, or switch to permitted bulk/partner sources.

### If hit rate is 5–20%
This is a realistic early result for free automated discovery. Use confidence filters and keep the attribution layer separate from Land Registry facts.

### If hit rate is >20%
Good signal. Increase `MAX_TRANSACTIONS_THIS_RUN` gradually: 25 → 100 → 250. Do not jump straight to thousands.

### Keep separate tables
- `pp_bn_standard_YYYY.parquet` = verified facts
- `agent_candidates_extracted.csv` = evidence snippets
- `agents_free_discovery.*` = canonical candidate agents
- `agent_transactions_free_discovery.*` = best candidate link per transaction

Never overwrite verified Land Registry data with inferred agent fields.